In [ ]:
pip install pandas pyarrow

In [71]:
pip install odfpy pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 717.0/717.0 kB 38.0 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for odfpy: filename=odfpy-1.4.1-py2.py3-none-any.whl size=160716 sha256=1852bb3ab60cdf01d3e99ff8645973d8d5ad094d1a8eeb3f15198500e3e601fe
  Stored in directory: /home/onyxia/.cache/pip/wheels/8e/cd/9f/979443982946991080916064e4c049b91941be1800825ff74b
Successfully built odfpy
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [odfpy]32m1/2 [odfpy]
Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd

# Chemin vers ton fichier feather
path_to_file = "/home/onyxia/stage_2A/data/classifications_core.feather"
path_to_file1="/home/onyxia/stage_2A/data/financials.feather"
path_to_file2="/home/onyxia/stage_2A/data/isin_codes.feather"
path_to_file3="/home/onyxia/stage_2A/data/legal_info.feather"
path_to_file4="/home/onyxia/stage_2A/data/names_1types.feather"


# Lecture du fichier
classif = pd.read_feather(path_to_file)
financials=pd.read_feather(path_to_file1)
isin_codes=pd.read_feather(path_to_file2)
legal_info=pd.read_feather(path_to_file3)
names=pd.read_feather(path_to_file4)

In [84]:
# On charge le fichier complet sans choisir d'onglet (sheet_name=None)
fichier_complet = pd.read_excel('data/etalab-donnees-membres-2014.ods', engine='odf', sheet_name=None)

# On affiche la liste des onglets trouvés
noms_onglets = list(fichier_complet.keys())
print("Onglets trouvés dans le fichier :", noms_onglets)

Onglets trouvés dans le fichier : ['Methodologie', 'Descriptif_des_variables', 'Entreprises_membres_en_2014', 'Entreprises_Ens_region_2014', 'Orga_Form_Labo_Ens_membres_2014']


In [85]:
# Remplace 'Liste_Entreprises' par le vrai nom que tu as vu s'afficher
df_vrai = pd.read_excel('data/etalab-donnees-membres-2014.ods', engine='odf', sheet_name='Entreprises_membres_en_2014')
print("Nouvelle taille :", df_vrai.shape)

Nouvelle taille : (71, 151)


In [95]:


# 1. Forcer l'affichage de TOUTES les lignes (pas de coupure au milieu)
pd.set_option('display.max_rows', None)

# 2. Forcer l'affichage de TOUTES les colonnes
pd.set_option('display.max_columns', None)

# 3. Forcer l'affichage de tout le texte sans le tronquer (ex: les longs noms de pôles)
pd.set_option('display.max_colwidth', None)


In [96]:
df_vrai[['numpole', 'designation_pole','nb_eti_2014']]

,numpole,designation_pole,nb_eti_2014
0,20051023,Atlanpole Biothérapies,8
1,20051408,Optitec,8
2,20051569,Aquimer,11
3,20051578,Cap Digital Paris-Région,36
4,20051617,Industries et Agro-Ressources,40
5,20051661,Pôle Européen de la Céramique,13
6,20051701,Hippolia,S
7,20051705,Pôle Nucléaire Bourgogne,33
8,20051767,Plastipolis,44
9,20051948,Cosmetic Valley,47


on filtre les entreprises françaises

In [16]:
financials=financials[financials['country_code']=="FR"]
names=names[names['country_code']=="FR"]
legal_info=legal_info[legal_info['country_code']=="FR"]
isin_codes=isin_codes[isin_codes['country_code']=="FR"]
classif=classif[classif['country_code']=="FR"]

on supprime les variables avec des valeurs manquantes

In [44]:
# On crée une copie propre en supprimant les lignes où les variables financières reines sont NaN
financials_clean = financials.dropna(subset=['total_assets', 'sales', 'operating_revenue_turnover'], how='any')

# On vérifie la taille du nouveau dataset
print("Nouvelle taille du dataset :", len(financials_clean))
print("Pourcentage de lignes conservées :", (len(financials_clean) / len(financials)) * 100)

Nouvelle taille du dataset : 16544089
Pourcentage de lignes conservées : 93.99620642351302


In [54]:

merge0 = pd.merge(
    financials_clean, 
    names, 
    on=['bvdid', 'country_code'],   
    how='left'      
)


In [59]:
legal_info.head(50)

,bvdid,country_code,status,status_date,standardised_legal_form,national_legal_form,date_of_incorporation,type_of_entity,category_of_the_company,listed_delisted_unlisted,main_exchange,delisted_date,ipo_date,information_provider
3,542051180,FR,Active,NaN,Public limited companies,European company,195401.0,Corporate,Very large company,Listed,Euronext Paris,NaN,NaN,Ellisphere
31,552100554,FR,Active,NaN,Public limited companies,Limited company with managing body,195501.0,Corporate,Very large company,Listed,Euronext Paris,NaN,200101.0,Ellisphere
33,652014051,FR,Active,NaN,Public limited companies,Limited company - SA,196301.0,Corporate,Very large company,Listed,Euronext Paris,NaN,197001.0,Ellisphere
39,552081317,FR,Active,NaN,Public limited companies,Limited company - SA,195501.0,Corporate,Very large company,Listed,Euronext Paris,NaN,200511.0,Ellisphere
42,542065479,FR,Active,NaN,Public limited companies,Limited company - SA,195401.0,Corporate,Very large company,Unlisted,Unlisted,NaN,NaN,Ellisphere
45,383474814,FR,Active,NaN,Private limited companies,"Limited company, simplified - SAS",199110.0,Corporate,Very large company,Unlisted,Unlisted,NaN,NaN,Ellisphere
46,542062559,FR,Dissolved,201806.0,Companies with unknown/unrecorded legal form,,188001.0,Corporate,Very large company,Delisted,Delisted,201006.0,NaN,World'Vest Base Inc.
52,542107651,FR,Active,NaN,Public limited companies,Limited company - SA,195401.0,Corporate,Very large company,Listed,Euronext Paris,NaN,200507.0,Ellisphere
53,429660863,FR,Active,NaN,Public limited companies,Limited company - SA,200002.0,Corporate,Very large company,Unlisted,Unlisted,NaN,NaN,Ellisphere
54,410409460,FR,Active,NaN,Private limited companies,"Limited company, simplified - SAS",199701.0,Corporate,Very large company,Unlisted,Unlisted,NaN,NaN,Ellisphere


In [57]:
names.head(50)

,bvdid,country_code,name,entity_type
14417368,*0000341758,FR,GROUP PMG FINANCE,F
14417369,*0000361620,FR,AZALIYA COMPANY,C
14417370,*0000362720,FR,GLOBAL SECURITIES HOUSE FRANCE COMPANY SAS,E
14417371,*0000365242,FR,SETRIM EN CHATREUSE,C
14417372,*0000378772,FR,TE FRANCE,C
14417373,*10019582,FR,Q.S.P SYSTEMS,C
14417374,*100461,FR,SCI IPB (Immobilière Pantin Bobigny),C
14417375,*100466,FR,STE FIAL,C
14417376,*1010850,FR,FAMILLE GUERBET,I
14417377,*1010857,FR,Famille BUCHER,I


In [61]:
merge1 = pd.merge(
    merge0, 
    legal_info, 
    on=['bvdid','country_code'],
    how='left'      
)

In [64]:
merge2 = pd.merge(
    merge1, 
    classif, 
    on=['bvdid','country_code'],
    how='left'      
)

In [66]:
merge2.head()

,bvdid,country_code,consolidation_code,filing_type,closing_date,number_of_months,original_units,original_currency,exchange_rate_from_original_currency,total_assets,...,date_of_incorporation,type_of_entity,category_of_the_company,listed_delisted_unlisted,main_exchange,delisted_date,ipo_date,information_provider,bvd_major_sector,nace_rev_2_core_code_4_digits
0,00367EF,FR,C1,Annual report,19971231,12,thousands,EUR,1.0,6.106193e+09,...,183501.0,Corporate,Very large company,Unlisted,Unlisted,NaN,NaN,World'Vest Base Inc.,Other services,7311.0
1,00367EF,FR,C1,Annual report,19961231,12,thousands,EUR,1.0,5.992771e+09,...,183501.0,Corporate,Very large company,Unlisted,Unlisted,NaN,NaN,World'Vest Base Inc.,Other services,7311.0
2,00367EF,FR,C1,Annual report,19951231,12,thousands,EUR,1.0,5.201560e+09,...,183501.0,Corporate,Very large company,Unlisted,Unlisted,NaN,NaN,World'Vest Base Inc.,Other services,7311.0
3,00367EF,FR,C1,Annual report,19941231,12,thousands,EUR,1.0,4.356536e+09,...,183501.0,Corporate,Very large company,Unlisted,Unlisted,NaN,NaN,World'Vest Base Inc.,Other services,7311.0
4,00367EF,FR,C1,Annual report,19931231,12,thousands,EUR,1.0,4.061394e+09,...,183501.0,Corporate,Very large company,Unlisted,Unlisted,NaN,NaN,World'Vest Base Inc.,Other services,7311.0


In [67]:

# 1. On convertit la colonne au format Date en spécifiant le format d'origine '%Y%m%d'
# (%Y = année à 4 chiffres, %m = mois à 2 chiffres, %d = jour à 2 chiffres)
merge2['closing_datetime'] = pd.to_datetime(merge2['closing_date'], format='%Y%m%d', errors='coerce')
# 2. On extrait l'année très facilement
merge2['year'] = merge2['closing_datetime'].dt.year

# On regarde le résultat
print(merge2[['closing_date', 'year']].head())

   closing_date  year
0      19971231  1997
1      19961231  1996
2      19951231  1995
3      19941231  1994
4      19931231  1993


In [68]:
merge2['year'].value_counts()

year
2012    1138514
2013    1114944
2011    1082752
2010    1015935
2009     951490
2008     924046
2014     888813
2007     879217
2006     833517
2005     788150
2004     753516
2015     753008
2003     721218
2002     669426
2001     630668
2016     588995
2000     584141
1999     541732
1998     492113
1997     432397
1996     392318
1995     223528
2017     141367
1994        356
1993        300
1992        263
1991        253
1990        249
1989        235
1988        183
2018        170
1987        104
1986         77
1985         54
1984         40
Name: count, dtype: int64

In [99]:
import pandas as pd

# 1. Lien direct vers l'export CSV officiel des membres des pôles franciliens
url_api = "https://data.iledefrance.fr/api/explore/v2.1/catalog/datasets/les-entreprises-membres-des-poles-de-competitivite-franciliens/exports/csv"

print("📥 Téléchargement de la base des pôles depuis l'Open Data...")
df_api = pd.read_csv(url_api, sep=';')

# 2. On filtre pour ne garder QUE le pôle Systematic
# (Le nom exact dans leur base peut varier, on utilise une recherche textuelle large)
df_systematic = df_api[df_api['pole'].str.contains('Systematic', case=False, na=False)].copy()

# 3. Nettoyage de sécurité sur les SIREN (on force à 9 chiffres)
df_systematic['siren_clean'] = df_systematic['siren'].astype(str).str.strip().str.zfill(9)

# On supprime les doublons de SIREN s'il y en a
liste_siren_systematic = df_systematic['siren_clean'].unique()

print(f"✅ Extraction réussie ! Nombre d'entreprises uniques trouvées pour Systematic : {len(liste_siren_systematic)}")
print("\nAperçu des premières entreprises trouvées :")
print(df_systematic[['nom_entreprise', 'siren_clean']].head(10))

📥 Téléchargement de la base des pôles depuis l'Open Data...


HTTPError: HTTP Error 404: Not Found